In [ ]:
println("Loading imports")
include("../src/naml.jl")

In [ ]:
println("Initialising data")

function pseudorandom_padic_int(p, prec, exp)
    K = PadicField(p, prec)
    coeffs = rand(0:(p-1), exp+1)
    return K(sum([coeffs[i] * p^(i-1) for i in Base.eachindex(coeffs)]))
end

# Initialise the p-adic field
prec = 20
p = 2
K = PadicField(p, prec)
poly_degree = 7

c = [pseudorandom_padic_int(p, prec, 5) for i in 1:poly_degree]
vars = ["x"]
append!(vars, ["a_"*string(i) for i in 1:poly_degree])
pts = [ValuationPolydisc{PadicFieldElem, Int}([c_i], [prec]) for c_i in c]
data = [(p, 0) for p in pts]
@show vars
# We're trying to fit the data points (x, y) = {(c1, 0), (c2, 0), (c3, 0)}
# using the function g_a(x) = (x-a)(x-2a)(x-4a)
R, v =  polynomial_ring(K, vars)

# # Initialise g_a(x) using a wrapper for absolute polynomials and their sums
g = PolydiscFunction([prod([(v[1]-v[i+1]) for i in 1:poly_degree])])
# Specify that x is the data variable and a is the parameter variable
model = AbstractModel(g, [(i == 1) for i in 1:(poly_degree+1)])

In [ ]:
println("Initialising optimisation tools")
state = 1
config = (true, 0)
# Let's pick the Gauss point as our initial parameter
gauss_point = ValuationPolydisc([K(1) for i in 1:poly_degree], zeros(Int, poly_degree))
# ℓ is the loss function of this model wrt the L¹ norm 
ℓ = MPE_loss_init(model, data, 1)
# We optimise using Greedy descent
greedy_optim = greedy_descent_init(gauss_point, ℓ, state, config)

In [ ]:
println("Optimising parameter")
# Compute the loss 

println("Loss before starting greedy descent is ", eval_loss(greedy_optim))

# Make 20 steps using greedy descent
N_epochs = 20
t1 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(greedy_optim))
    step!(greedy_optim)
end 
t2 = time()

# Compute the new value of the loss
println("Greedy descent finished in ", t2-t1, " seconds. \nThe Final parameter a is ", greedy_optim.param)

# Now we compute the distances from parameters to roots of the polynomials. Notice that these might not come in the same orders
# so we need to take that into account when computing the distances. 

# First find the closest param to each root (for simplicity we allow repetitions in the params)
dist_arr = [minimum([padic_abs(greedy_optim.param.center[i] - c[j]) for i in Base.eachindex(c)]) for j in Base.eachindex(c)]

# Then take max over that 
println("Distance of param to roots is at most ", maximum(dist_arr))

In [ ]:
# # Declare new model with same starting point as the first one
# new_model = Model(f, gauss_point)
# # Now let's optimise using gradient descent 
# gradient_optim = gradient_descent_init(data, new_model, ℓ, 3)

# println("Loss before starting gradient descent is ", eval_loss(gradient_optim))

# t3 = time()
# for i in 1:N_epochs
#     println("Loss at epoch ", i, " is ", eval_loss(gradient_optim))
#     step!(gradient_optim)
# end 
# t4 = time()

# # Compute the new value of the loss
# println("Gradient descent finished in ", t4-t3, "Seconds. \nThe Final parameter a is ", gradient_optim.model.param)